# Preprocesamiento

Recorre paso a paso lo que hace `src/preprocess.py`: construcción del dataset limpio desde los .xlsx crudos, versionado y la clase `Preprocess` (split, inválidos, outliers, transformación, escalado y faltantes). Las funciones viven en el .py; acá solo se llaman en orden para ver qué hace cada una.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from preprocess import *

GUARDAR = False    # True: escribe data/clean/df_total.xlsx
VERSIONAR = False  # True: además lo registra en data/logs/data_log.json

## 1. Procesar cada Excel

Por cada archivo crudo:
- `juntar_kpi`: agrega los registros del día de la hoja *Consolidado KPI* (`mean`) en la fila de cierre.
- `limpiar`: se queda con el cierre del día (>= 23:30) y descarta los días incompletos (< `REGISTROS_DIA` registros).
- `elegir_mas_similar`: si una fecha está repetida, se queda con la fila más parecida al día anterior y al siguiente.
- `unir_hojas`: une todas las hojas por `fecha`.

`procesar_excel` hace todo eso para **un** archivo.

In [2]:
partes = [procesar_excel(ruta, crit_kpi="mean") for ruta in RAW_DEFAULT]
for ruta, parte in zip(RAW_DEFAULT, partes):
    print(f"{ruta.name}: {parte.shape} | {parte.index.min().date()} -> {parte.index.max().date()}")

Totalizadores Planta - 2021_2023.xlsx: (594, 396) | 2021-07-02 -> 2023-03-05
Totalizadores Planta 2020_2022.xlsx: (604, 315) | 2020-07-01 -> 2022-03-15
Totalizadores Planta 2022_2023.xlsx: (469, 396) | 2022-07-01 -> 2023-10-25


## 2. Juntar los Excel

Los archivos se solapan en fechas; `juntar_datasets` los combina con `combine_first` (si un día falta o está incompleto en un archivo, se toma del otro).

In [3]:
df = juntar_datasets(partes)
df.shape

(1178, 413)

## 3. Columnas vacías y tipos

Se borran las columnas con muchos nulos o casi sin varianza y se pasan a numéricas las columnas de texto.

In [4]:
df, eliminadas = borrar_columnas_vacias(df, min_nulos=1100, umbral_varianza=10)
df = convertir_object_a_numerico(df)
print(f"Columnas eliminadas: {len(eliminadas)} | shape: {df.shape}")

Columnas eliminadas: 148 | shape: (1178, 265)


## 4. Completar los días faltantes

Para que el `shift` del target sea siempre al día siguiente, cada día del calendario que falta se completa con el promedio del día válido anterior y el siguiente.

In [5]:
dias_antes = len(df)
df = completar_dias(df)
print(f"Días agregados: {len(df) - dias_antes} | días consecutivos: "
      f"{(df[COL_FECHA].diff().dropna() == pd.Timedelta(days=1)).all()}")

Días agregados: 34 | días consecutivos: True


## 5. Features externas: clima y mes cíclico

- Temperatura y humedad (media, máx, mín) de Tecate desde Open-Meteo, con caché en `data/clima/`.
- Mes como `mes_sin` / `mes_cos`, así diciembre queda al lado de enero.

Estas columnas (`COLS_SIN_LIMITES`) no las tocan los pasos de inválidos y outliers.

In [6]:
df = agregar_clima(df)
df = agregar_mes_ciclico(df)
df[[COL_FECHA, *COLS_SIN_LIMITES]].describe().round(2)

,fecha,temp media (°c),temp max (°c),temp min (°c),humedad media (%),humedad max (%),humedad min (%),mes_sin,mes_cos
count,1212,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00
mean,2022-02-26 12:00:00,17.67,24.30,11.87,55.57,78.45,33.39,-0.06,-0.06
min,2020-07-01 00:00:00,3.20,6.20,-4.30,12.00,19.00,4.00,-1.00,-1.00
25%,2021-04-29 18:00:00,12.00,18.20,6.70,41.00,65.00,19.00,-0.87,-0.87
50%,2022-02-26 12:00:00,17.10,24.40,11.40,56.00,86.00,31.00,0.00,-0.00
75%,2022-12-26 06:00:00,23.42,30.80,17.10,72.00,95.00,45.00,0.50,0.50
max,2023-10-25 00:00:00,34.80,44.00,26.10,98.00,100.00,89.00,1.00,1.00
std,NaN,6.83,7.71,6.52,19.78,20.24,17.87,0.70,0.71


## 6. Variable objetivo

`frio_mañana (kw)` es `frio (kw)` del día siguiente; el último día queda sin target.

In [7]:
df = agregar_target(df)
df[[COL_FECHA, "frio (kw)", TARGET]].tail()

,fecha,frio (kw),frio_mañana (kw)
1207,2023-10-21,9481.0,11587.0
1208,2023-10-22,11587.0,10123.0
1209,2023-10-23,10123.0,10360.0
1210,2023-10-24,10360.0,10568.0
1211,2023-10-25,10568.0,NaN


## 7. Guardar y versionar

`construir_dataset(...)` hace las secciones 1 a 6 en una sola llamada. Con `versionar=True` registra el hash del archivo en `data_log.json` (solo si cambió).

In [8]:
if GUARDAR:
    df.to_excel(RUTA_LIMPIO, index=False)
    if VERSIONAR:
        registrar_version(RUTA_LIMPIO, ["preprocesamiento.ipynb"], comentario="...")

# equivalente en una línea:
# df = construir_dataset(guardar=RUTA_LIMPIO, versionar=True, comentario="...")
# o, si ya está construido:
# df = cargar_dataset_limpio()

## 8. Configuración del preprocesamiento

Cada paso se configura con su dataclass y se desactiva pasando `None`:

| Paso | Config | Opciones |
|---|---|---|
| Inválidos | `ConfigInvalidos` | `no_negativas` (`"ee"` por default), `max_abs`, `rangos`; acción `nan` / `clip` / `delete` |
| Outliers | `ConfigOutliers` | `iqr` / `zscore_mod` / `percentil`; acción `nan` / `clip` / `delete`; `excluir` |
| Transformación | `ConfigTransformer` | `yeo-johnson` / `boxcox` / `log` / `sqrt` |
| Escalado | `ConfigScaler` | `standard` / `robust` / `minmax` |
| Faltantes | `ConfigFaltantes` | `mean` / `median` / `interpolate` / `knn` |
| Split | `ConfigSplit` | `test_size` (corte temporal) |

Todo se aprende solo con train y se aplica igual a test.

In [9]:
pp = Preprocess(
    invalidos=ConfigInvalidos(accion="nan"),
    outliers=ConfigOutliers(metodo="iqr", accion="nan"),
    transformer=ConfigTransformer(metodo="yeo-johnson"),
    scaler=ConfigScaler(metodo="robust"),
    faltantes=ConfigFaltantes(estrategia="knn", n_vecinos=5),
    split=ConfigSplit(test_size=0.15),
    verbose=True,
)

## 9. Train / test split

Corte por fecha: test son los últimos días, para no predecir el pasado con el futuro.

In [10]:
train, test = pp.train_test_split(df)
print(f"train: {train.shape} {train[COL_FECHA].min().date()} -> {train[COL_FECHA].max().date()}")
print(f"test:  {test.shape} {test[COL_FECHA].min().date()} -> {test[COL_FECHA].max().date()}")

train: (1030, 274) 2020-07-01 -> 2023-04-26
test:  (182, 274) 2023-04-27 -> 2023-10-25


## 10. Fit en train y transform en train y test

Orden: filas sin target → inválidos → outliers → transformación → escalado → faltantes.

In [11]:
train_pp = pp.fit_transform(train)
test_pp = pp.transform(test, nombre="test")

X_train, y_train = pp.separar_xy(train_pp)
X_test, y_test = pp.separar_xy(test_pp)
print(X_train.shape, X_test.shape, "| NaN:", X_train.isna().sum().sum() + X_test.isna().sum().sum())

# todo junto: X_train, X_test, y_train, y_test = pp.run(df)

  -> [fit] train (1030, 274)
  -> [fit] inválidos (nan): 130 valores, quedan 1030 filas


  -> [fit] outliers (iqr, nan): 11406 valores, quedan 1030 filas


  -> [fit] transformer yeo-johnson entrenado
  -> [fit] scaler robust entrenado
  -> [fit] imputador knn entrenado, 0 columnas descartadas por nulos
  -> [train] inicio (1030, 274)
  -> [train] filas sin target eliminadas: 0
  -> [train] inválidos (nan): 130 valores, quedan 1030 filas
  -> [train] outliers (iqr, nan): 11406 valores, quedan 1030 filas
  -> [train] transformer yeo-johnson aplicado
  -> [train] scaler robust aplicado


  -> [train] faltantes (knn): 26569 valores imputados
  -> [train] fin (1030, 272)
  -> [test] inicio (182, 274)
  -> [test] filas sin target eliminadas: 1
  -> [test] inválidos (nan): 7 valores, quedan 181 filas
  -> [test] outliers (iqr, nan): 2832 valores, quedan 181 filas
  -> [test] transformer yeo-johnson aplicado
  -> [test] scaler robust aplicado


  -> [test] faltantes (knn): 4806 valores imputados
  -> [test] fin (181, 272)
(1030, 272) (181, 272) | NaN: 0


## 11. Resumen

Configuración usada y cuántos valores tocó cada paso.

In [12]:
pd.DataFrame(pp.resumen()["pasos"])

,train,test
filas_inicio,1030,182
invalidos_valores,130,7
outliers_valores,11406,2832
transformer,yeo-johnson,yeo-johnson
scaler,robust,robust
columnas_descartadas_por_nulos,0,0
faltantes_valores,26569,4806
filas_fin,1030,181
